<h1><center> ST7-OSTP : TP0 </center></h1>
<h2><center> 09/02/2021 </center></h2>

Ce TP a pour objectif de vous familiariser à la résolution de Programmes Linéaires (PL) avec le solveur Gurobi en utilisant le module $\texttt{gurobipy}$ de Python.

Ce TP est en deux parties :

* La **Partie 1** présente trois méthodes d'instanciation et de résolution appliquées à deux PL.  <br>
* La **Partie 2** vise à vous faire pratiquer ces trois méthodes.

## Modules

In [ ]:
# Modules de base
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Module relatif à Gurobi
from gurobipy import *

# 1. Présentation de trois méthodes pour instancier et résoudre un PL

 ## 1.1. Méthode 1 : par ajout, une par une, des variables et des contraintes

### 1.1.1. Application de la méthode sur le premier programme $(P_1)$

On considère le PL $\boldsymbol{(P_1)}$ suivant : 

$$
\boldsymbol{(P_1)} \qquad
\begin{array}{cc}
    \underset{v_0, ~ v_1}{\text{max}} & v_1 \\
    \text{s.c.} & \left\{
    \begin{array}{rcrlc}
        v_0 & - & v_1 & \leq 4 & \boldsymbol{(C_1)}  \\
        v_0 & + & v_1 & \leq 4 & \boldsymbol{(C_2)} \\
        -0.25 v_0 & + & v_1 & \leq 1 & \boldsymbol{(C_3)}
    \end{array}  \right. \\
    & v_0, ~ v_1 \geq 0
\end{array}
$$

#### Instanciation du PL

In [ ]:
# -- Initialisation du modèle --
# m : Model
m = Model("PL1")


# -- Ajout des variables continues  --
# v0 : Var
v0 = m.addVar(vtype = GRB.CONTINUOUS, name = "v0", lb = 0)
# v1 : Var
v1 = m.addVar(vtype = GRB.CONTINUOUS, name = "v1", lb = 0)
# Plus d'informations ici: 
# https://www.gurobi.com/documentation/9.1/refman/py_model_addvar.html#pythonmethod:Model.addVar


# -- Ajout des constraintes --
# c1 : Constr
c1 = m.addConstr(v0 - v1 <= 4, name = "C1")
# c2 : Constr
c2 = m.addConstr(v0 + v1 <= 4, name = "C2")
# c3 : Constr
c3 = m.addConstr(-0.25*v0 + v1 <= 1, name = "C3")
# Plus d'informations ici:
# https://www.gurobi.com/documentation/9.1/refman/py_model_addconstr.html


# -- Ajout de la fonction objectif --
m.setObjective(v1, GRB.MAXIMIZE)
# Plus d'informations ici:
# https://www.gurobi.com/documentation/9.1/refman/py_model_setobjective.html


# -- Choix d'un paramétrage d'affichage minimaliste --
m.params.outputflag = 0 # mode muet
# Plus d'informations ici:
# https://www.gurobi.com/documentation/9.1/refman/outputflag.html


# -- Mise à jour du modèle  --
m.update()
# Plus d'informations ici: 
# https://www.gurobi.com/documentation/9.1/refman/py_model_update.html


# -- Affichage en mode texte du PL --
print(m.display())

#### Résolution du PL

In [ ]:
# -- Résolution --
m.optimize()
# Plus d'informations ici: 
# https://www.gurobi.com/documentation/9.1/refman/py_model_optimize.html


# -- Affichage de la solution --
print("La solution optimale est (v0, v1) = {} avec pour valeur de l'objectif z = {}".format((v0.x, v1.x), m.objVal))

#### Représentation graphique du PL (en bonus)

Comme le PL $\boldsymbol{(P_1)}$ possède deux variables, on peut en donner une représentation dans le plan $\mathbb{R}^2$.<br/>
Le code ci-dessous génère un graphique représentant dans $\mathbb{R}^2$ :<br/>
* les droites associées aux contraintes $\boldsymbol{(C_1)}$, $\boldsymbol{(C_2)}$ et $\boldsymbol{(C_3)}$ ;
* l'ensemble des solutions réalisables ;
* la solution optimale ;
* la fonction objectif à l'optimum.

In [ ]:
# -- Construction des droites associées aux contraintes --
v0s = np.linspace(-2, 8, 100)
v1s = np.linspace(-6, 6, 100)
# (C1)
v1s_C1 = v0s - 4
# (C2)
v1s_C2 = 4 - v0s
# (C3)
v1s_C3 = 1 + 0.25*v0s
# Borne sur v0
v0s_bound = 0*v1s
# Borne sur v1
v1s_bound = 0*v0s


# -- Dessin des droites associées aux contraintes --
plt.plot(v0s, v1s_C1, 'r', label = '$v_1 \geq v_0 - 4 \quad (C_1)$')
plt.plot(v0s, v1s_C2, 'b', label = '$v_1 \leq 4 - v_0 \quad (C_2)$')
plt.plot(v0s, v1s_C3, 'g', label = '$v_1 \leq 1 + 0.25v_0 \quad (C_3)$')
plt.plot(v0s_bound, v1s, 'y', label = "$v_0 \geq 0$")
plt.plot(v0s, v1s_bound, 'orange', label = "$v_1 \geq 0$")


# -- Construction de la droite associée à l'objectif --
v1s_obj = m.objVal*np.ones(100)


# -- Dessin de la droite associés à l'objectif --
plt.plot(v0s, v1s_obj, linestyle = '--', color = 'grey', label = "$v_1 = z^\star$")


# -- Dessin de la région des solutions réalisables --
v1s_sup = np.minimum(v1s_C2, v1s_C3)
v1s_inf = np.maximum(v1s_C1, v1s_bound)
plt.fill_between(v0s, v1s_sup, v1s_inf, where = (v1s_sup > v1s_inf) & (v0s > v0s_bound), 
                 color = 'grey', alpha = 0.3, label = "ens. des solutions réalisables")


# -- Dessin de la solution optimale --
plt.plot(v0.x, v1.x, 'ro', label = "$(v^\star_0, v^\star_1)$")


# -- Définition des paramètres du graphique --
plt.grid()
plt.xlim((-1, 7))
plt.ylim((-5, 5))
plt.xlabel('$v_0$')
plt.ylabel('$v_1$')
plt.legend(bbox_to_anchor = (1.05, 1), loc = 2, borderaxespad = 0.)
plt.title('Représentation graphique de $(P_1)$')


# -- Affichage du graphique --
plt.show()

### 1.1.2. Application de la méthode sur le deuxième programme $(P_2)$

On considère le PL $\boldsymbol{(P_2)}$ suivant : 

$$
\boldsymbol{(P_2)} \qquad
\begin{array}{cc}
    \underset{x_1, ~ x_2, ~ x_3}{\text{max}} & x_1 + 2 x_2 + 5 x_3 \\
    \text{s.c.} & \left\{
    \begin{array}{rcrcrlc}
        x_1 & - & x_2 & - & 3 x_3 & \geq 5 & \boldsymbol{(C_1)}  \\
        x_1 & + & 3 x_2 & - & 7 x_3 & \leq 10 & \boldsymbol{(C_2)} \\
        x_1 & & & & & \leq 10 & \boldsymbol{(C_3)}
    \end{array} \right. \\
    & x_1, ~ x_2, ~ x_3 \in \mathbb{R}^+
\end{array}
$$

#### Instanciation du PL

In [ ]:
# -- Initialisation du modèle --
m = Model("PL2")


# -- Ajout des variables continues  --
x1 = m.addVar(name = "x1", lb = 0, ub = 10)
x2 = m.addVar(name = "x2", lb = 0)
x3 = m.addVar(name = "x3", lb = 0)


# -- Ajout des constraintes --
m.addConstr(-x1 + x2 + 3*x3 <= -5, name = "C1")
m.addConstr(x1 + 3*x2 - 7*x3 <= 10, name = "C2")


# -- Ajout de la fonction objectif
m.setObjective(x1 + 2*x2 + 5*x3, GRB.MAXIMIZE)


# -- Choix d'un paramétrage d'affichage minimaliste --
m.params.outputflag = 0 # mode muet


# -- Mise à jour du modèle --
m.update()


# -- Affichage en mode texte du PL --
print(m.display())

#### Résolution du PL

In [ ]:
# -- Résolution --
m.optimize()


# -- Affichage des solutions --
print("Les solutions optimales sont")
print("x1 =", x1.x)
print("x2 =", x2.x)
print("x3 =", x3.x)
print("avec pour valeur de l'objectif z =", m.objVal)

 ## 1.2. Méthode 2 : par écriture matricielle

### 1.2.1. Application de la méthode sur le premier programme $(P_1)$

On considère à nouveau le premier PL $\boldsymbol{(P_1)}$: 

$$
\boldsymbol{(P_1)} \qquad
\begin{array}{cc}
    \underset{v_0, ~ v_1}{\text{max}} & v_1 \\
    \text{s.c.} & \left\{
    \begin{array}{rcrlc}
        v_0 & - & v_1 & \leq 4 & \boldsymbol{(C_1)}  \\
        v_0 & + & v_1 & \leq 4 & \boldsymbol{(C_2)} \\
        -0.25 v_0 & + & v_1 & \leq 1 & \boldsymbol{(C_3)}
    \end{array} \right. \\
    & v_0, ~ v_1 \geq 0
\end{array}
$$

#### Instanciation du PL

In [ ]:
# -- Initialisation du modèle --
# m : Model
m = Model("PL1_matrix")


# -- Ajout des variables continues comme vecteur de variables --
# V : MVar
V = m.addMVar(shape = 2, lb = 0, name = "v")
# Plus d'informations ici: 
# https://www.gurobi.com/documentation/9.1/refman/py_model_addmvar.html


# -- Ajout des contraintes comme produit matriciel --
# A : NumPy ndarray
A = np.array([ [1,    -1],
               [1,     1],
               [-0.25, 1] ])
# b : NumPy ndarray
b = np.array([4, 4, 1])
# C : MConstr
C = m.addConstr(A @ V <= b, name = "C")


# -- Ajout de la fonction objectif
# obj : NumPy ndarray
obj = np.array([0.0, 1.0])
m.setObjective(obj @ V, GRB.MAXIMIZE)


# -- Choix d'un paramétrage d'affichage minimaliste --
m.params.outputflag = 0 # mode muet
# Plus d'informations ici:
# https://www.gurobi.com/documentation/9.1/refman/outputflag.html


# -- Mise à jour du modèle  --
m.update()
# Plus d'informations ici: 
# https://www.gurobi.com/documentation/9.1/refman/py_model_update.html


# -- Affichage en mode texte du PL --
print(m.display())

#### Résolution du PL

In [ ]:
# -- Résolution --
m.optimize()


# -- Affichage de la solution --
print("La solution optimale est [v0, v1] = {} avec pour valeur de l'objectif z = {}".format(V.x, m.objVal))

### 1.2.2. Application de la méthode sur le deuxième programme $(P_2)$

On considère à nouveau le deuxième PL $\boldsymbol{(P_2)}$: 

$$
\boldsymbol{(P_2)} \qquad
\begin{array}{cc}
    \underset{x_1, ~ x_2, ~ x_3}{\text{max}} & x_1 + 2 x_2 + 5 x_3 \\
    \text{s.c.} & \left\{
    \begin{array}{rcrcrlc}
        x_1 & - & x_2 & - & 3 x_3 & \geq 5 & \boldsymbol{(C_1)}  \\
        x_1 & + & 3 x_2 & - & 7 x_3 & \leq 10 & \boldsymbol{(C_2)} \\
        x_1 & & & & & \leq 10 & \boldsymbol{(C_3)}
    \end{array} \right. \\
    & x_1, ~ x_2, ~ x_3 \in \mathbb{R}^+
\end{array}
$$

#### Instanciation du PL

In [ ]:
# -- Installation du modèle --
# m : Model
m = Model("PL2_2")


# -- Ajout des variables continues comme vecteur de variables --
X = m.addMVar(shape = 3, name = "X")


# -- Ajout des contraintes comme produit matriciel  --
A = np.array([ [-1, 1, 3],
               [ 1, 3,-7],
               [ 1, 0, 0] ])
b = np.array([-5, 10, 10])
m.addConstr(A @ X <= b, name = "C")


# -- Ajout de la fonction objectif --
obj = np.array([1, 2, 5])
m.setObjective(obj @ X, GRB.MAXIMIZE)


# -- Choix d'un paramétrage d'affichage minimaliste --
m.params.outputflag = 0 # mode muet


# -- Mise à jour du modèle --
m.update()


# -- Affichage en mode texte du PL --
print(m.display())

#### Résolution du PL

In [ ]:
# -- Résolution --
m.optimize()


# -- Affichage de la solution --
print("La solution optimale est (x1, x2, x3) = {} avec pour valeur de l'objectif z = {}".format(list(X.x), m.objVal))

## 1.3. Méthode 3 : par lecture d'un fichier .lp

### Application de la méthode sur le deuxième programme $(P_2)$

Consulter le fichier $\texttt{model.lp}$ fourni et analyser les lignes d'instructions de la cellule de code ci-dessous pour comprendre comment s'effectue le chargement du $\boldsymbol{(P_2)}$ depuis un fichier $\texttt{.lp}$.

#### Instanciation du PL

In [ ]:
# -- Chargement du PL depuis un fichier .lp --
m = read('model.lp')


# -- Choix d'un paramétrage d'affichage minimaliste  --
m.params.outputflag = 0 # mode muet


# -- Affichage en mode texte du PL --
print(m.display())

#### Résolution du PL

In [ ]:
# -- Résolution --
m.optimize()


# -- Extraction de la solution et affichage des valeurs optimales --
x_values = [var.x for var in m.getVars()]
print("La solution optimale est [x1, x2, x3] = {} avec pour objectif z = {}".format(x_values, m.objVal))

# 2. Mise en pratique : à vous de jouer maintenant !

Implémenter et résoudre suivant chacune des 3 méthodes détaillées ci-dessus les programmes linéaires ci-dessous.

$$
\boldsymbol{(P_0)} \qquad
\begin{array}{cc}
    \underset{x_1, ~ x_2}{\text{max}} & x_1 + x_2 \\
    \text{s.c.} & \left\{
    \begin{array}{rcrl}
        x_1 & + & x_2 & \geq 14  \\
        - x_1 & & & \leq 8 \\
        & - & x_2 & \leq 10
    \end{array} \right. \\
    & x_1, ~ x_2 \in \mathbb{R}^+
\end{array}
$$

Que remarquez-vous ? 

Consultez [la documentation en ligne](https://www.gurobi.com/documentation/9.1/refman/optimization_status_codes.html#sec:StatusCodes) et l'[exemple suivant](https://www.gurobi.com/documentation/9.1/examples/lp_py.html) pour savoir comment éviter ce problème. 

In [ ]:
# Corrigé

# -- Initialisation du modèle --
# m : Model
m = Model("P0")

# -- Ajout des variables continues  --
# x1 : Var
x1 = m.addVar(vtype = GRB.CONTINUOUS, name = "x1", lb = 0)
# x2 : Var
x2 = m.addVar(vtype = GRB.CONTINUOUS, name = "x2", lb = 0)


# -- Ajout des constraintes --
# c1 : Constr
c1 = m.addConstr(x1 + x2 >= 14, name = "C1")
# c2 : Constr
c2 = m.addConstr(-x1 <= 8, name = "C2")
# c3 : Constr
c3 = m.addConstr(-x2 <= 10, name = "C3")


# -- Ajout de la fonction objectif --
m.setObjective(x1 + x2, GRB.MAXIMIZE)


# -- Choix d'un paramétrage d'affichage minimaliste --
m.params.outputflag = 0 # mode muet


# -- Mise à jour du modèle  --
m.update()


# -- Affichage en mode texte du PL --
print(m.display())

In [ ]:
# -- Résolution --
m.optimize()


# -- Vérification du statut et Affichage (le cas échéant) des solutions --
if m.status == GRB.INF_OR_UNBD:
    m.setParam(GRB.Param.Presolve, 0)
    m.optimize()

if m.status == GRB.INFEASIBLE:
    print(m.display(), "\n\tN'A PAS DE SOLUTION!!!")
elif m.status == GRB.UNBOUNDED:
    print(m.display(), "\n\tEST NON BORNÉ!!!")
else:
    print("Les solutions optimales sont")
    print("x1 =", x1.x)
    print("x2 =", x2.x)
    print("avec pour valeur de l'objectif z =", m.objVal)

$$
\boldsymbol{(P_1)} \qquad
\begin{array}{cc}
    \underset{x_1, ~ x_2}{\text{max}} & x_1 + x_2 \\
    \text{s.c.} & \left\{
    \begin{array}{rcrl}
        2 x_1 & + & x_2 & \leq 14  \\
        - x_1 & + & 2 x_2 & \leq 8 \\
        2 x_1 & - & x_2 & \leq 10
    \end{array} \right. \\
    & x_1, ~ x_2 \in \mathbb{R}^+
\end{array}
$$

In [ ]:
# Corrigé

# -- Initialisation du modèle --
# m : Model
m = Model("P1")

# -- Ajout des variables continues  --
# x1 : Var
x1 = m.addVar(vtype = GRB.CONTINUOUS, name = "x1", lb = 0)
# x2 : Var
x2 = m.addVar(vtype = GRB.CONTINUOUS, name = "x2", lb = 0)


# -- Ajout des constraintes --
# c1 : Constr
c1 = m.addConstr(2*x1 + x2 <= 14, name = "C1")
# c2 : Constr
c2 = m.addConstr(-x1 + 2*x2 <= 8, name = "C2")
# c3 : Constr
c3 = m.addConstr(2*x1 - x2 <= 10, name = "C3")


# -- Ajout de la fonction objectif --
m.setObjective(x1 + x2, GRB.MAXIMIZE)


# -- Choix d'un paramétrage d'affichage minimaliste --
m.params.outputflag = 0 # mode muet


# -- Mise à jour du modèle  --
m.update()


# -- Affichage en mode texte du PL --
print(m.display())

In [ ]:
# -- Résolution --
m.optimize()


# -- Vérification du statut et Affichage (le cas échéant) des solutions --
if m.status == GRB.INF_OR_UNBD:
    m.setParam(GRB.Param.Presolve, 0)
    m.optimize()

if m.status == GRB.INFEASIBLE:
    print(m.display(), "\n\tN'A PAS DE SOLUTION!!!")
elif m.status == GRB.UNBOUNDED:
    print(m.display(), "\n\tEST NON BORNÉ!!!")
else:
    print("Les solutions optimales sont")
    print("x1 =", x1.x)
    print("x2 =", x2.x)
    print("avec pour valeur de l'objectif z =", m.objVal)

$$
\boldsymbol{(P_2)} \qquad
\begin{array}{cc}
    \underset{x_1, ~ x_2, ~ x_3}{\text{min}} & 14 x_1 + 8 x_2 + 10 x_3 \\
    \text{s.c.} &  \left\{
    \begin{array}{rcrcrl}
        2 x_1 & - & x_2 & + & 2 x_3 & \geq 1  \\
        x_1 & + & 2 x_2 & - & x_3 & \geq 1
    \end{array} \right. \\
    & x_1, ~ x_2, ~ x_3 \in \mathbb{R}^+
\end{array}
$$

In [ ]:
# Corrigé

# -- Initialisation du modèle --
# m : Model
m = Model("P2_matrix")


# -- Ajout des variables continues comme vecteur de variables --
# V : MVar
X = m.addMVar(shape = 3, lb = 0, name = "x")


# -- Ajout des contraintes comme produit matriciel --
# A : NumPy ndarray
A = np.array([ [2,    -1,     2],
               [1,     2,    -1] ])
# b : NumPy ndarray
b = np.array([1, 1])
# C : MConstr
C = m.addConstr(A @ X >= b, name = "C")


# -- Ajout de la fonction objectif
# obj : NumPy ndarray
obj = np.array([14, 8, 10])
m.setObjective(obj @ X, GRB.MINIMIZE)


# -- Choix d'un paramétrage d'affichage minimaliste --
m.params.outputflag = 0 # mode muet


# -- Mise à jour du modèle  --
m.update()

# -- Affichage en mode texte du PL --
print(m.display())

In [ ]:
# -- Résolution --
m.optimize()


# -- Vérification du statut et Affichage (le cas échéant) des solutions --
if m.status == GRB.INF_OR_UNBD:
    m.setParam(GRB.Param.Presolve, 0)
    m.optimize()

if m.status == GRB.INFEASIBLE:
    print(m.display(), "\n\tN'A PAS DE SOLUTION!!!")
elif m.status == GRB.UNBOUNDED:
    print(m.display(), "\n\tEST NON BORNÉ!!!")
else:
    print("La solution optimale est (x1, x2, x3) = {} avec pour valeur de l'objectif z = {}".format(list(X.x),
                                                                                                    m.objVal))

$$
\boldsymbol{(P_3)} \qquad
\begin{array}{cc}
    \underset{A, ~ B, ~ C}{\text{max}} & 160 A + 300 B + 500 C \\
    \text{s.c.} & \left\{
    \begin{array}{rcrcrl}
        3 A & + & 3.5 B & + & 5 C & \leq 1800 \\
        4 A & + & 5 B & + & 8 C & \leq 2400 \\
        A & + & 1.5 B & + & 8 C & \leq 720 \\
        A & & & & & \geq 100 \\
        & & B & & & \geq 150 \\
        & & & & C & \geq 75
    \end{array} \right. \\
    & A, ~ B, ~ C \in \mathbb{N}
\end{array}
$$

In [ ]:
# Corrigé

# -- Initialisation du modèle --
# m : Model
m = Model("P3_matrix")


# -- Ajout des variables continues comme vecteur de variables --
# V : MVar
X = m.addMVar(shape = 3, lb = 0, vtype=GRB.INTEGER)

# Maj des noms des variables
X[0].VarName = "A"
X[1].VarName = "B"
X[2].VarName = "C"

# -- Ajout des contraintes comme produit matriciel --
# M : NumPy ndarray
M = np.array([ [ 3,    3.5,     5],
               [ 4,      5,     8],
               [ 1,    1.5,     3],
               [-1,      0,     0],
               [ 0,     -1,     0],
               [ 0,      0,    -1] ])
# b : NumPy ndarray
b = np.array([1800, 2400, 720, -100, -150, -75])
# C : MConstr
C = m.addConstr(M @ X <= b, name = "C")


# -- Ajout de la fonction objectif
# obj : NumPy ndarray
obj = np.array([160, 300, 500])
m.setObjective(obj @ X, GRB.MAXIMIZE)


# -- Choix d'un paramétrage d'affichage minimaliste --
m.params.outputflag = 0 # mode muet


# -- Mise à jour du modèle  --
m.update()

# -- Affichage en mode texte du PL --
print(m.display())

In [ ]:
# -- Résolution --
m.optimize()


# -- Vérification du statut et Affichage (le cas échéant) des solutions --
if m.status == GRB.INF_OR_UNBD:
    m.setParam(GRB.Param.Presolve, 0)
    m.optimize()

if m.status == GRB.INFEASIBLE:
    print(m.display(), "\n\tN'A PAS DE SOLUTION!!!")
elif m.status == GRB.UNBOUNDED:
    print(m.display(), "\n\tEST NON BORNÉ!!!")
else:
    print("La solution optimale est (A, B, C) = {} avec pour valeur de l'objectif z = {}".format(list(X.x),
                                                                                                    m.objVal))

$$
\boldsymbol{(P_4)} \qquad
\begin{array}{cc}
    \underset{x}{\text{max}} & c^T x \\
    \text{s.c.} & A x \leq b \\
    & x \in \left(\mathbb{R}^+\right)^2
\end{array}
$$

avec
$$
A = \begin{pmatrix} 1 & 1 \\ 1 & 3 \\ 0 & 3 \end{pmatrix}, \qquad
b = \begin{pmatrix} 150 \\ 210 \\ 180 \end{pmatrix}, \qquad
c = \begin{pmatrix} 300 \\ 500 \end{pmatrix}
$$

In [ ]:
# Corrigé

# -- Initialisation du modèle --
# m : Model
m = Model("P4")

# -- Ajout des variables continues  --
# x1 : Var
x1 = m.addVar(vtype = GRB.CONTINUOUS, name = "x1", lb = 0)
# x2 : Var
x2 = m.addVar(vtype = GRB.CONTINUOUS, name = "x2", lb = 0)


# -- Ajout des constraintes --
# c1 : Constr
c1 = m.addConstr(x1 + x2 <= 150, name = "C1")
# c2 : Constr
c2 = m.addConstr(x1 + 3*x2 <= 210, name = "C2")
# c3 : Constr
c3 = m.addConstr(x2 <= 180, name = "C3")


# -- Ajout de la fonction objectif --
m.setObjective(300 *x1 + 500*x2, GRB.MAXIMIZE)


# -- Choix d'un paramétrage d'affichage minimaliste --
m.params.outputflag = 0 # mode muet


# -- Mise à jour du modèle  --
m.update()


# -- Affichage en mode texte du PL --
print(m.display())

In [ ]:
# -- Résolution --
m.optimize()


# -- Vérification du statut et Affichage (le cas échéant) des solutions --
if m.status == GRB.INF_OR_UNBD:
    m.setParam(GRB.Param.Presolve, 0)
    m.optimize()

if m.status == GRB.INFEASIBLE:
    print(m.display(), "\n\tN'A PAS DE SOLUTION!!!")
elif m.status == GRB.UNBOUNDED:
    print(m.display(), "\n\tEST NON BORNÉ!!!")
else:
    print("Les solutions optimales sont")
    print("x1 =", x1.x)
    print("x2 =", x2.x)
    print("avec pour valeur de l'objectif z =", m.objVal)